##### ARTI 560 - Computer Vision

## Visual Representations with DINOv2 - Exercise

### Exercise 1: Unsupervised Clustering

In this exercise, you will use the `KMeans` algorithm from sklearn to group 20 images from the Oxford Pet dataset into 2 clusters (Cats vs. Dogs) based purely on their CLS tokens.

Instructions:

1.  Extract the 384-dimensional [CLS] tokens from 20 images of the Oxford-IIIT Pet dataset. Ensure your selection includes a mix of both cats and dogs.

2. Apply K-Means Clustering ($n=2$) to group the vectors based on mathematical similarity rather than provided labels.

3. Compare the predicted clusters against ground-truth labels.

In [3]:
import os
import torch
import numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
from sklearn.cluster import KMeans
from collections import Counter

# 1. Setup device and load DINOv2 base model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "facebook/dinov2-small"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# 2. Path to Kaggle Dataset
image_dir = "/kaggle/input/datasets/tanlikesmath/the-oxfordiiit-pet-dataset/images"
all_filenames = os.listdir(image_dir)

# Oxford Pet dataset convention: 
# Filenames starting with a Capital letter are Cats. Lowercase letters are Dogs.
cat_files = [f for f in all_filenames if f[0].isupper() and f.endswith('.jpg')]
dog_files = [f for f in all_filenames if f[0].islower() and f.endswith('.jpg')]

# Collect 10 of each
selected_cats = cat_files[:10]
selected_dogs = dog_files[:10]

images_to_process = selected_cats + selected_dogs
# Ground Truth: 0 for Cats, 1 for Dogs
ground_truth = [0] * 10 + [1] * 10 

# 3. Extract [CLS] tokens using DINOv2
cls_tokens = []

with torch.no_grad():
    for fname in images_to_process:
        img_path = os.path.join(image_dir, fname)
        img = Image.open(img_path).convert("RGB")
        
        # Preprocess and forward pass
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model(**inputs)
        
        # Pull pooler_output (the [CLS] token representation)
        cls_token = outputs.pooler_output.squeeze(0).cpu().numpy()
        cls_tokens.append(cls_token)

cls_tokens = np.array(cls_tokens)

# 4. Apply K-Means Clustering (n=2)
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
predicted_clusters = kmeans.fit_predict(cls_tokens)

# 5. Evaluate and Print Results
print("--- Evaluation ---")
print(f"Ground Truth: {ground_truth} (First 10 are Cats [0], Next 10 are Dogs [1])")
print(f"KMeans Preds: {list(predicted_clusters)}")

cluster_0_labels = [ground_truth[i] for i in range(20) if predicted_clusters[i] == 0]
cluster_1_labels = [ground_truth[i] for i in range(20) if predicted_clusters[i] == 1]

print("\nCluster Distribution Analysis:")
print(f"Images grouped into Cluster 0 belong to original labels: {Counter(cluster_0_labels)}")
print(f"Images grouped into Cluster 1 belong to original labels: {Counter(cluster_1_labels)}")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

--- Evaluation ---
Ground Truth: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] (First 10 are Cats [0], Next 10 are Dogs [1])
KMeans Preds: [np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(1)]

Cluster Distribution Analysis:
Images grouped into Cluster 0 belong to original labels: Counter({1: 8})
Images grouped into Cluster 1 belong to original labels: Counter({0: 10, 1: 2})


### Exercise 2: Image Classification with DINOv2

In this exercise you'll use a DINOv2 model with a pre-trained linear head to classify an image. You will observe how the model maps visual features to specific ImageNet-1k categories.

Instructions:
1. For this exercise, you must use the following Model ID. This specific checkpoint includes the necessary classification head trained on ImageNet-1k:

    Model ID: `facebook/dinov2-small-imagenet1k-1-layer`

2. Find an image online to make the inference. To ensure the model has a fair chance of success, the image should belong to one of the ImageNet-1k classes (e.g., a Golden Retriever, a grand piano, a school bus, or a coffee mug).

In [4]:
import torch
import requests
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

# 1. Load the specific DINOv2 model with the ImageNet classification head
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "facebook/dinov2-small-imagenet1k-1-layer"

processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModelForImageClassification.from_pretrained(model_id).to(device)
model.eval()

# 2. Grab a target image online (A Golden Retriever)
url = "https://images.unsplash.com/photo-1552053831-71594a27632d?w=500"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# 3. Preprocess and run inference
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# 4. Map the highest index to its ImageNet label string
predicted_class_idx = logits.argmax(-1).item()
predicted_label = model.config.id2label[predicted_class_idx]

# Calculate confidence percentage
probs = torch.nn.functional.softmax(logits, dim=-1)
confidence = probs[0][predicted_class_idx].item() * 100

print("--- Inference Results ---")
print(f"Predicted Class: {predicted_label}")
print(f"Confidence Score: {confidence:.2f}%")

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/91.3M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/225 [00:00<?, ?it/s]

--- Inference Results ---
Predicted Class: golden retriever
Confidence Score: 91.38%
